In [25]:
# math librairies
import numpy as np
from scipy import ndimage

import numpy.linalg as alg

from plotly import express as px
import pandas               # used with plotly

# 2D image processing 
import cv2

# custom librairies
import imicpe
import imicpe.optim as optim
print(imicpe.__version__)

1.0.14


# **Initialisation**

## Chargement de la vérité terrain

In [26]:
# génération d'une vérité terrain 1D xbar
N = 100
dom  = np.arange(1,N+1)
#xbar = np.zeros((N,),float)
#xbar[25:50] =  np.sin(.25*dom[25:50])
#xbar[60:70] = -1.
#xbar[75:90] =  1.

In [27]:
# # chargement d'une image 
xbar = cv2.imread('.\images\parrot-gray.png',0)/255.

## Création de la donnée dégradée

In [28]:
# choix de l'opérateur d'acquisition
# Utiliser l'opérateur identité compatible avec la forme de xbar
def H(x):
    return x

def Ht(x):
    return x

# création de la donnée z, dégradée par un bruit blanc gaussien d'écart-type sig
sig = 0.05
# bruit de même forme que xbar (évite les erreurs de diffusion/matricielle)
b = sig * np.random.randn(*xbar.shape)
z = H(xbar) + b

## Fonction de coût

In [29]:
# attache aux données
def f(x):
    return np.sum((H(xbar) - z)**2)

# régularisation
def R(x):
    return np.sum((x - xbar)**2)

# fonction de coût globale
def E(x,lam):
    return f(x) + lam * R(x)


# **Algorithme**

In [30]:
# paramètres du modèle


# paramètres de l'algo
Niter = 100                                 # nombre max d'itérations
tk=0.5

# initialisation des variables
En = np.zeros((Niter,),float) * np.nan
xn = np.zeros(xbar.shape)                                    # x0



# itérations
for i in range(1,Niter):

    grad = 2 * Ht(H(xn) - z)
    xn =xn -tk*grad
    En[i] = E(xn,0)
    

xhat = xn

## **Affichage des resultats**

In [31]:
# plot cost function
plt_cost = pandas.DataFrame({'x':np.arange(Niter), 'y':En, 'legend':'cost', 'type':'cost_function'})

fig = px.line(plt_cost,
              x='x', 
              y='y', 
              log_x=True, log_y=True,
              labels={'x':'itérations (logscale)','y':'E(x^k) (logscale)'},
              title='Évolution de la fonction de coût en fonction des itérations',
              width=800, height=350)
fig.show()

In [32]:
# plot signals (for 2D images: plot the central row)
row_idx = xbar.shape[0] // 2
dom_col = np.arange(1, xbar.shape[1] + 1)

plt_xbar = pandas.DataFrame({'x': dom_col, 'y': xbar[row_idx, :], 'legend': 'xbar'})
plt_z    = pandas.DataFrame({'x': dom_col, 'y': z[row_idx, :],    'legend': 'z'})
plt_xhat = pandas.DataFrame({'x': dom_col, 'y': xhat[row_idx, :], 'legend': 'xhat'})

data = pandas.concat([plt_xbar, plt_z, plt_xhat], ignore_index=True)

ymin = 1.1 * np.amin([xbar[row_idx, :], z[row_idx, :], xhat[row_idx, :]])
ymax = 1.1 * np.amax([xbar[row_idx, :], z[row_idx, :], xhat[row_idx, :]])

fig = px.line(data,
              x='x',
              y='y',
              color='legend',
              color_discrete_sequence=['black', 'orange', 'orangered'],
              line_dash='legend', line_dash_sequence=['dot', 'solid', 'dash'],
              labels={'x': 'indices', 'y': ''},
              title='Signaux (ligne centrale)',
              range_y=[ymin, ymax],
              width=800, height=450)
fig.show()



In [33]:
 # plot images
plt_img = np.array([xbar,z,xhat]) 
legend   = ['xbar', 'z', 'xhat']
nb_img_per_row = 3
fig = px.imshow(plt_img, color_continuous_scale='gray', range_color=[0,1], 
                title='Résultats',
                facet_col=0, facet_col_spacing=0, facet_col_wrap=nb_img_per_row,
                width=700, height=400,
                )
fig.update_layout(coloraxis_showscale=True)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
item_map={f'{i}':key for i, key in enumerate(legend)}
fig.for_each_annotation(lambda a: a.update(text=item_map[a.text.split("=")[1]]))
fig.show()